# 01 — Data Understanding

**IT3051 – Fundamentals of Data Mining | Mini Project 2026**

This notebook examines the raw **Hotel Booking Demand Dataset** to answer: **What data do we actually have?** The binary classification target is `is_canceled`: **0 = Not Cancelled**, **1 = Cancelled**.

We inspect the dataset's structure, recorded values, and basic descriptive summaries before any cleaning or transformation. No preprocessing or feature-selection decisions are made at this stage. **The raw dataset must remain unchanged.**

## 2 — Imports and project path

Only pandas, standard-library path handling, and notebook display utilities are needed. The root check supports execution from either the repository root or its `notebooks` directory. The dataset is always located using `data/raw/hotel_bookings.csv`.

The pandas version is reported because dtype inference can vary across versions.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
project_root = next(
    (candidate for candidate in (cwd, cwd.parent)
     if (candidate / "requirements.txt").is_file()
     and (candidate / "notebooks").is_dir()
     and (candidate / "data" / "raw").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory.")

data_path = project_root / "data" / "raw" / "hotel_bookings.csv"
if not data_path.is_file():
    raise FileNotFoundError("Place hotel_bookings.csv inside data/raw/ before running this notebook.")

print(f"Dataset: {data_path.relative_to(project_root).as_posix()}")
print(f"pandas version: {pd.__version__}")

Dataset: data/raw/hotel_bookings.csv
pandas version: 3.0.3


## 3 — Load raw data

Read the CSV into `df` with pandas' default inference. Subsequent cells inspect `df` without assigning new columns, changing values, or saving a dataset.

In [2]:
df = pd.read_csv(data_path)
print(f"Successfully loaded {len(df):,} rows and {len(df.columns)} columns.")

Successfully loaded 119,390 rows and 32 columns.


## 4 — Dataset dimensions

Each row represents a recorded hotel booking. Each column contains an attribute of that booking or its recorded outcome. The column count includes the target; it is not a count of selected model predictors.

In [3]:
print(f"Observations: {df.shape[0]:,}")
print(f"Columns (including target): {df.shape[1]}")
print(f"df.shape: {df.shape}")

Observations: 119,390
Columns (including target): 32
df.shape: (119390, 32)


## 5 — Preview raw records

The first five records provide a structural preview of field values and layout. They are not treated as representative evidence about all bookings.

In [4]:
with pd.option_context("display.max_columns", None):
    display(df.head())

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## 6 — Column inventory

List every column in source order and verify the target and hotel grouping column before inspecting their distributions.

In [5]:
column_inventory = pd.DataFrame({
    "Number": range(1, len(df.columns) + 1),
    "Column": df.columns,
})
display(column_inventory)
print(f"Total columns: {len(df.columns)}")

missing_required = sorted({"is_canceled", "hotel"} - set(df.columns))
if missing_required:
    raise ValueError(f"Required columns missing from the CSV: {missing_required}")
print("Required columns verified: is_canceled, hotel.")

,Number,Column
0,1,hotel
1,2,is_canceled
2,3,lead_time
3,4,arrival_date_year
4,5,arrival_date_month
5,6,arrival_date_week_number
6,7,arrival_date_day_of_month
7,8,stays_in_weekend_nights
8,9,stays_in_week_nights
9,10,adults


Total columns: 32
Required columns verified: is_canceled, hotel.


## 7 — Data types

The table records the inferred pandas dtype and non-null count for each column. Numeric and non-numeric/categorical-like groups below are initial dtype-based groups only. **Pandas dtype alone does not determine a feature's eventual treatment**: an integer may represent a category or an identifier, and a text column may contain dates. No final preprocessing classifications are assigned.

In [6]:
type_summary = pd.DataFrame({
    "Column": df.columns,
    "pandas dtype": [str(df[column].dtype) for column in df.columns],
    "Non-null count": [df[column].count() for column in df.columns],
})
display(type_summary)
numeric_columns = df.select_dtypes(include="number").columns.tolist()
categorical_columns = df.select_dtypes(exclude="number").columns.tolist()
print(f"Numeric columns ({len(numeric_columns)}): {', '.join(numeric_columns)}")
print(f"Non-numeric/categorical-like columns ({len(categorical_columns)}): {', '.join(categorical_columns)}")

,Column,pandas dtype,Non-null count
0,hotel,str,119390
1,is_canceled,int64,119390
2,lead_time,int64,119390
3,arrival_date_year,int64,119390
4,arrival_date_month,str,119390
5,arrival_date_week_number,int64,119390
6,arrival_date_day_of_month,int64,119390
7,stays_in_weekend_nights,int64,119390
8,stays_in_week_nights,int64,119390
9,adults,int64,119390


Numeric columns (20): is_canceled, lead_time, arrival_date_year, arrival_date_week_number, arrival_date_day_of_month, stays_in_weekend_nights, stays_in_week_nights, adults, children, babies, is_repeated_guest, previous_cancellations, previous_bookings_not_canceled, booking_changes, agent, company, days_in_waiting_list, adr, required_car_parking_spaces, total_of_special_requests
Non-numeric/categorical-like columns (12): hotel, arrival_date_month, meal, country, market_segment, distribution_channel, reserved_room_type, assigned_room_type, deposit_type, customer_type, reservation_status, reservation_status_date


## 8 — Basic numerical summary

Descriptive statistics show the observed count, centre, spread, and range of numerically stored attributes, including numeric codes and the target. Statistics on codes do not establish a quantitative interpretation. Unusual minimum or maximum values will be investigated during dedicated data-quality and EDA stages; they are not labelled as errors or transformed here.

In [7]:
display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
is_canceled,119390.0,0.370416,0.482918,0.00,0.00,0.000,1.0,1.0
lead_time,119390.0,104.011416,106.863097,0.00,18.00,69.000,160.0,737.0
arrival_date_year,119390.0,2016.156554,0.707476,2015.00,2016.00,2016.000,2017.0,2017.0
arrival_date_week_number,119390.0,27.165173,13.605138,1.00,16.00,28.000,38.0,53.0
arrival_date_day_of_month,119390.0,15.798241,8.780829,1.00,8.00,16.000,23.0,31.0
stays_in_weekend_nights,119390.0,0.927599,0.998613,0.00,0.00,1.000,2.0,19.0
stays_in_week_nights,119390.0,2.500302,1.908286,0.00,1.00,2.000,3.0,50.0
adults,119390.0,1.856403,0.579261,0.00,2.00,2.000,2.0,55.0
children,119386.0,0.103890,0.398561,0.00,0.00,0.000,0.0,10.0
babies,119390.0,0.007949,0.097436,0.00,0.00,0.000,0.0,10.0


## 9 — Basic categorical summary

For each non-numeric column, report its number of unique non-null values and a most frequent non-null value with its count. Ties use the first value returned by pandas. This compact table avoids printing every category and leaves recorded categories unchanged.

In [8]:
categorical_rows = []
for column in categorical_columns:
    counts = df[column].value_counts(dropna=True)
    categorical_rows.append({
        "Feature": column,
        "Unique non-null values": df[column].nunique(dropna=True),
        "Most frequent value": counts.index[0] if len(counts) else pd.NA,
        "Frequency": int(counts.iloc[0]) if len(counts) else 0,
    })
display(pd.DataFrame(categorical_rows))

,Feature,Unique non-null values,Most frequent value,Frequency
0,hotel,2,City Hotel,79330
1,arrival_date_month,12,August,13877
2,meal,5,BB,92310
3,country,177,PRT,48590
4,market_segment,8,Online TA,56477
5,distribution_channel,5,TA/TO,97870
6,reserved_room_type,10,A,85994
7,assigned_room_type,12,A,74053
8,deposit_type,3,No Deposit,104641
9,customer_type,4,Transient,89613


## 10 — Target variable understanding

Inspect unique target values and count each class. Percentages use **all records** as the denominator, with any null or unexpected values retained and explicitly labelled in the summary. Labels are added only to the presentation table; the raw target stays unchanged.

This describes the observed class distribution. Class imbalance will be formally investigated later; no rebalancing decision is made here.

In [9]:
print("Unique target values:", df["is_canceled"].unique().tolist())
target_labels = {0: "0 = Not Cancelled", 1: "1 = Cancelled"}

def target_label(value):
    if pd.isna(value):
        return "Missing target"
    return target_labels.get(value, f"Unexpected target: {value}")

target_counts = df["is_canceled"].value_counts(dropna=False).sort_index()
target_summary = pd.DataFrame({
    "Class": [target_label(value) for value in target_counts.index],
    "Count": target_counts.to_numpy(),
    "Percentage of all records": (target_counts / len(df) * 100).round(2).to_numpy(),
})
display(target_summary)

Unique target values: [0, 1]


,Class,Count,Percentage of all records
0,0 = Not Cancelled,75166,62.96
1,1 = Cancelled,44224,37.04


## 11 — Hotel variable understanding

Inspect the hotel categories and their shares of all records. The cross-tabulation reports raw counts by hotel type and target value; row-wise cancellation percentages are not needed at this stage. Null groups, if present, are retained with `dropna=False`.

This initial descriptive comparison motivates the City Hotel versus Resort Hotel investigation. It does not establish statistical significance, causation, or better performance from separate models. Whether hotel-specific modelling improves predictive performance remains a hypothesis for later experiments.

In [10]:
print("Unique hotel categories:", df["hotel"].unique().tolist())
hotel_counts = df["hotel"].value_counts(dropna=False)
hotel_summary = pd.DataFrame({
    "Hotel": hotel_counts.index,
    "Booking count": hotel_counts.to_numpy(),
    "Percentage of all records": (hotel_counts / len(df) * 100).round(2).to_numpy(),
})
display(hotel_summary)

hotel_target_counts = pd.crosstab(df["hotel"], df["is_canceled"], dropna=False)
display(hotel_target_counts.rename(columns=target_label))

Unique hotel categories: ['Resort Hotel', 'City Hotel']


,Hotel,Booking count,Percentage of all records
0,City Hotel,79330,66.45
1,Resort Hotel,40060,33.55


is_canceled,0 = Not Cancelled,1 = Cancelled
hotel,,
City Hotel,46228,33102
Resort Hotel,28938,11122


## 12 — Cardinality overview

Cardinality is the number of distinct non-null values. Presenting every column in source order helps identify binary, low-cardinality, high-cardinality, and continuous-looking attributes for later investigation. Cardinality alone does not determine a feature's role or preprocessing needs.

In [11]:
cardinality_summary = pd.DataFrame({
    "Column": df.columns,
    "dtype": [str(df[column].dtype) for column in df.columns],
    "Unique non-null values": [df[column].nunique(dropna=True) for column in df.columns],
})
display(cardinality_summary)

,Column,dtype,Unique non-null values
0,hotel,str,2
1,is_canceled,int64,2
2,lead_time,int64,479
3,arrival_date_year,int64,3
4,arrival_date_month,str,12
5,arrival_date_week_number,int64,53
6,arrival_date_day_of_month,int64,31
7,stays_in_weekend_nights,int64,17
8,stays_in_week_nights,int64,35
9,adults,int64,14


## 13 — Initial dataset observations

The factual summary below is generated from the inspected data so that it updates when the notebook is rerun. Missing-value treatment, duplicate removal, unusual-value/outlier decisions, leakage auditing, and preprocessing belong to later stages. No cleaning, engineered features, visual EDA, training/testing split, or modelling has been performed.

In [12]:
observations = [
    f"The dataset contains **{df.shape[0]:,} rows and {df.shape[1]} columns**, including the target.",
    "Observed target values: " + ", ".join(target_label(value) for value in target_counts.index) + ".",
    "Observed hotel categories: " + ", ".join(str(value) for value in hotel_counts.index) + ".",
    f"Pandas infers **{len(numeric_columns)} numeric** and **{len(categorical_columns)} non-numeric/categorical-like** columns; these are not final modelling classifications.",
]
if {"City Hotel", "Resort Hotel"}.issubset(set(df["hotel"].unique())):
    observations.append("Both **City Hotel** and **Resort Hotel** are present.")
for value, count in target_counts.items():
    observations.append(f"**{target_label(value)}**: {count:,} records ({count / len(df) * 100:.2f}% of all records).")
display(Markdown("\n".join(f"- {item}" for item in observations)))

- The dataset contains **119,390 rows and 32 columns**, including the target.
- Observed target values: 0 = Not Cancelled, 1 = Cancelled.
- Observed hotel categories: City Hotel, Resort Hotel.
- Pandas infers **20 numeric** and **12 non-numeric/categorical-like** columns; these are not final modelling classifications.
- Both **City Hotel** and **Resort Hotel** are present.
- **0 = Not Cancelled**: 75,166 records (62.96% of all records).
- **1 = Cancelled**: 44,224 records (37.04% of all records).